# Environment Setup
## Library Imports

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import timm
import wandb
from kaggle_secrets import UserSecretsClient
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## Weights and Biases Configuration Setup

In [2]:
user_secrets = UserSecretsClient()

os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

os.environ["WANDB_DISABLE_SERVICE"] = "true"
os.environ["WANDB_START_METHOD"] = "thread"

try:
    wandb.login()
    WANDB_MODE = "online"
except:
    WANDB_MODE = "disabled" 
    print("Running in offline mode - WandB logging disabled")

wandb.login()
WANDB_PROJECT = "23f3003805-t12026"

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Device Configuration

In [3]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Tesla T4


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data Configuration
## Loading data

In [5]:
DATA_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"


STEMS_DIR = f"{DATA_DIR}/genres_stems"
MASHUPS_DIR = f"{DATA_DIR}/mashups"
ESC50_DIR = f"{DATA_DIR}/ESC-50-master/audio"

TEST_CSV = f"{DATA_DIR}/test.csv"
SUBMISSION_CSV = f"{DATA_DIR}/sample_submission.csv"

## Genre Label Mapping 

In [6]:
GENRES = sorted(os.listdir(STEMS_DIR))
GENRE_TO_IDX = {g:i for i,g in enumerate(GENRES)}
IDX_TO_GENRE = {i:g for g,i in GENRE_TO_IDX.items()}
EPOCHS=40

print(GENRES)

['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


# Dataset Construction
## Stem Metadata Creation

In [7]:
records = []

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)

    for song in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song)

        stems = {
            "bass": os.path.join(song_path,"bass.wav"),
            "drums": os.path.join(song_path,"drums.wav"),
            "other": os.path.join(song_path,"other.wav"),
            "vocals": os.path.join(song_path,"vocals.wav")
        }

        records.append({
            "genre":genre,
            "genre_id":GENRE_TO_IDX[genre],
            "stems":stems
        })

df = pd.DataFrame(records)
df.head()

,genre,genre_id,stems
0,blues,0,{'bass': '/kaggle/input/jan-2026-dl-gen-ai-pro...
1,blues,0,{'bass': '/kaggle/input/jan-2026-dl-gen-ai-pro...
2,blues,0,{'bass': '/kaggle/input/jan-2026-dl-gen-ai-pro...
3,blues,0,{'bass': '/kaggle/input/jan-2026-dl-gen-ai-pro...
4,blues,0,{'bass': '/kaggle/input/jan-2026-dl-gen-ai-pro...


## ESC-50 Noise Dataset

In [8]:
noise_files = []

for f in os.listdir(ESC50_DIR):
    if f.endswith(".wav"):
        noise_files.append(os.path.join(ESC50_DIR,f))

print("Noise files:",len(noise_files))

Noise files: 2000


# Audio Utility Functions
## Audio Loading Function

In [9]:
def load_audio(path, sr=22050):

    y, _ = librosa.load(path, sr=sr)

    return y

## Synthetic Mashup Generator

In [10]:
genre_groups = {g: df[df.genre == g] for g in GENRES}

def synthetic_mashup(genre):

    songs = genre_groups[genre].sample(4)

    bass = load_audio(songs.iloc[0].stems["bass"])
    drums = load_audio(songs.iloc[1].stems["drums"])
    vocals = load_audio(songs.iloc[2].stems["vocals"])
    other = load_audio(songs.iloc[3].stems["other"])

    min_len = min(len(bass), len(drums), len(vocals), len(other))

    bass = bass[:min_len]
    drums = drums[:min_len]
    vocals = vocals[:min_len]
    other = other[:min_len]

    # random mixing weights (VERY important)
    w = np.random.uniform(0.5, 1.5, 4)

    mix = (
        w[0]*bass +
        w[1]*drums +
        w[2]*vocals +
        w[3]*other
    )

    mix = mix / (np.max(np.abs(mix)) + 1e-6)

    return mix

# Audio Data Augmentation
## Noise Injection

In [11]:
def add_noise(signal, snr_db=10):

    noise_path = random.choice(noise_files)
    noise = load_audio(noise_path)

    if len(noise) < len(signal):
        noise = np.tile(noise, len(signal)//len(noise)+1)

    noise = noise[:len(signal)]

    signal_power = np.mean(signal**2)
    noise_power = np.mean(noise**2)

    factor = np.sqrt(signal_power/(10**(snr_db/10)*noise_power))

    noisy = signal + factor*noise

    return noisy

## Time Stretching and Padding

In [12]:
def augment_audio(y, target_len=None):

    if random.random() < 0.5:
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y, rate=rate)

    if random.random() < 0.5:
        y = add_noise(y, snr_db=random.randint(5, 20))

    if target_len is not None:

        if len(y) > target_len:
            y = y[:target_len]

        else:
            pad = target_len - len(y)
            y = np.pad(y, (0, pad))

    return y

## Random Audio Cropping

In [13]:
def random_crop(y, crop_size):

    if len(y) <= crop_size:
        return np.pad(y, (0, crop_size - len(y)))

    start = random.randint(0, len(y) - crop_size)

    return y[start:start + crop_size]

# Dataset & DataLoader
## Custom PyTorch Dataset

In [14]:
class StemDataset(Dataset):

    def __init__(self, df, sr=22050):

        self.df = df
        self.sr = sr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        y = synthetic_mashup(row.genre)

        y = random_crop(y, 22050 * 6)
        y = augment_audio(y, 22050 * 6)

        y = torch.tensor(y).float().unsqueeze(0)

        label = row.genre_id

        return y, label

## Custom Collate Function

In [15]:
def collate_fn(batch):

    waves = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    max_len = max([w.shape[1] for w in waves])

    padded = []

    for w in waves:
        pad = max_len - w.shape[1]
        padded.append(nn.functional.pad(w,(0,pad)))

    return torch.stack(padded), torch.tensor(labels)

## Train-Validation Split

In [16]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df.genre
)

## DataLoader Construction

In [17]:
train_ds = StemDataset(train_df)
val_ds = StemDataset(val_df)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

# Spectrogram Feature Transformation
## Mel Spectrogram Conversion

In [18]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
    f_min=20,
    f_max=11025
)

# Feature Engineering — MFCC Baseline
## MFCC Feature Extraction

In [19]:
def extract_mfcc_from_stems(row):

    y = synthetic_mashup(row.genre)
    y = random_crop(y, 22050 * 6)

    mfcc = librosa.feature.mfcc(y=y, sr=22050, n_mfcc=40)

    return mfcc.mean(axis=1)

print("Extracting MFCC features...")

Extracting MFCC features...


## MFCC Dataset Generation

In [20]:
X = []
y = []

for i,row in tqdm(df.iterrows(), total=len(df)):
    
    feat = extract_mfcc_from_stems(row)
    
    X.append(feat)
    y.append(row.genre_id)

X = np.array(X)
y = np.array(y)

X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size=0.1,stratify=y
)

100%|██████████| 1000/1000 [07:03<00:00,  2.36it/s]


# Baseline Model — Logistic Regression

In [21]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="logistic_regression_baseline",
    reinit=True, 
    config={
        "model": "LogisticRegression",
        "features": "MFCC",
        "max_iter": 2000
    }
)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_084614-u4cjwhr5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run logistic_regression_baseline
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/u4cjwhr5


## Training Logistic Regression

In [22]:
clf = LogisticRegression(max_iter=2000)

clf.fit(X_train,y_train)

pred = clf.predict(X_val)

f1 = f1_score(y_val,pred,average="macro")
acc = accuracy_score(y_val,pred)

run.log({
    "F1_score":f1,
    "accuracy":acc
})

print("Logistic Regression")
print("F1:",f1)
print("Accuracy:",acc)

run.finish()

wandb: updating run metadata


Logistic Regression
F1: 0.44454189220241175
Accuracy: 0.45


wandb: 
wandb: Run history:
wandb: F1_score ▁
wandb: accuracy ▁
wandb: 
wandb: Run summary:
wandb: F1_score 0.44454
wandb: accuracy 0.45
wandb: 
wandb: 🚀 View run logistic_regression_baseline at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/u4cjwhr5
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_084614-u4cjwhr5/logs


# CNN Model
## CNN Architecture

In [23]:
class SimpleCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.net = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128,len(GENRES))

    def forward(self,x):

        # x = x.squeeze(1)
        x = self.mel(x)
        # x = x.unsqueeze(1)
        x = torch.log(x + 1e-6)

        x = self.net(x)

        x = x.view(x.size(0),-1)

        return self.fc(x)

## CNN Training

In [24]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="simple_cnn",
    reinit=True,
    config={
        "model":"SimpleCNN",
        "optimizer":"Adam",
        "lr":1e-3,
        "epochs":10,
        "batch_size":64
    }
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_084620-6pnimh3m
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run simple_cnn
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/6pnimh3m


In [25]:
cnn_model = SimpleCNN().to(device)

optimizer = torch.optim.Adam(cnn_model.parameters(),lr=1e-3)

criterion = nn.CrossEntropyLoss()

for epoch in range(10):

    cnn_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x,y = x.to(device),y.to(device)

        optimizer.zero_grad()

        preds = cnn_model(x)

        loss = criterion(preds,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss/len(train_loader)

    run.log({
        "epoch":epoch,
        "train_loss":avg_loss
    })

    print("Epoch",epoch,"Loss",avg_loss)

100%|██████████| 15/15 [02:13<00:00,  8.90s/it]


Epoch 0 Loss 2.106456192334493


100%|██████████| 15/15 [01:53<00:00,  7.55s/it]


Epoch 1 Loss 1.892449434598287


100%|██████████| 15/15 [01:43<00:00,  6.87s/it]


...

Epoch 7 Loss 1.5730769236882527


100%|██████████| 15/15 [01:34<00:00,  6.33s/it]


Epoch 8 Loss 1.5418234904607138


100%|██████████| 15/15 [01:35<00:00,  6.33s/it]

Epoch 9 Loss 1.5128132661183675


## Model Evaluation

In [26]:
def evaluate_model(model):

    model.eval()

    preds_all=[]
    labels_all=[]

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(device)

            out=model(x)

            preds=out.argmax(1).cpu().numpy()

            preds_all.extend(preds)
            labels_all.extend(y.numpy())

    f1=f1_score(labels_all,preds_all,average="macro")
    acc=accuracy_score(labels_all,preds_all)

    return f1,acc

In [27]:
f1, acc = evaluate_model(cnn_model)

run.log({
    "epoch": epoch,
    "train_loss": avg_loss,
    "val_f1": f1,
    "val_accuracy": acc
})

print("CNN F1:", f1)
print("CNN Accuracy:", acc)

run.finish()

wandb: updating run metadata


CNN F1: 0.3862299599265543
CNN Accuracy: 0.43


wandb: 
wandb: Run history:
wandb:        epoch ▁▂▃▃▄▅▆▆▇██
wandb:   train_loss █▅▅▄▃▃▂▂▁▁▁
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb: 
wandb: Run summary:
wandb:        epoch 9
wandb:   train_loss 1.51281
wandb: val_accuracy 0.43
wandb:       val_f1 0.38623
wandb: 
wandb: 🚀 View run simple_cnn at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/6pnimh3m
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_084620-6pnimh3m/logs


# CRNN Model
## CRNN Architecture

In [28]:
class CRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.cnn = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256,len(GENRES))

    def forward(self,x):

        # x = x.squeeze(1)
        x = self.mel(x)
        # x = x.unsqueeze(1)

        x = torch.log(x+1e-6)

        x = self.cnn(x)

        x = x.mean(dim=2)

        x = x.permute(0,2,1)

        x,_ = self.gru(x)

        x = x.mean(dim=1)

        return self.fc(x)

## CRNN Training

In [29]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="crnn_model",
    reinit=True,
    config={
        "model":"CRNN",
        "optimizer":"Adam",
        "lr":3e-4,
        "epochs":12
    }
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_090337-f3geo916
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run crnn_model
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/f3geo916


In [30]:
crnn_model = CRNN().to(device)

optimizer = torch.optim.Adam(crnn_model.parameters(),lr=3e-4)

criterion = nn.CrossEntropyLoss()

for epoch in range(12):

    crnn_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x,y = x.to(device),y.to(device)

        optimizer.zero_grad()

        preds = crnn_model(x)

        loss = criterion(preds,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    run.log({
        "epoch":epoch,
        "train_loss":total_loss/len(train_loader),
        "learning_rate":optimizer.param_groups[0]['lr']
    })

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

100%|██████████| 15/15 [01:33<00:00,  6.25s/it]


Epoch 0 Loss 2.2762643814086916


100%|██████████| 15/15 [01:35<00:00,  6.34s/it]


Epoch 1 Loss 2.176631387074788


100%|██████████| 15/15 [01:34<00:00,  6.32s/it]


...

Epoch 9 Loss 1.756174119313558


100%|██████████| 15/15 [01:34<00:00,  6.30s/it]


Epoch 10 Loss 1.6432456096013388


100%|██████████| 15/15 [01:34<00:00,  6.33s/it]

Epoch 11 Loss 1.59668816725413


## Model Evaluation

In [31]:
f1, acc = evaluate_model(crnn_model)

run.log({
    "F1_score": f1,
    "accuracy": acc
})

print("CRNN F1:", f1)
print("CRNN Accuracy:", acc)

run.finish()

wandb: updating run metadata


CRNN F1: 0.3327307269772825
CRNN Accuracy: 0.38


wandb: 
wandb: Run history:
wandb:      F1_score ▁
wandb:      accuracy ▁
wandb:         epoch ▁▂▂▃▄▄▅▅▆▇▇█
wandb: learning_rate ▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train_loss █▇▇▆▆▅▄▃▂▃▁▁
wandb: 
wandb: Run summary:
wandb:      F1_score 0.33273
wandb:      accuracy 0.38
wandb:         epoch 11
wandb: learning_rate 0.0003
wandb:    train_loss 1.59669
wandb: 
wandb: 🚀 View run crnn_model at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/f3geo916
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_090337-f3geo916/logs


# EfficientNet Spectrogram Model
## EfficientNet Architecture

In [32]:
class EfficientNetAudio(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.freq_mask = torchaudio.transforms.FrequencyMasking(24)
        self.time_mask = torchaudio.transforms.TimeMasking(40)

        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    def forward(self,x):

        # x = x.squeeze(1)
        x = self.mel(x)
        # x = x.unsqueeze(1)

        x = torch.log(x+1e-6)

        x = (x - x.mean(dim=(2,3),keepdim=True)) / (x.std(dim=(2,3),keepdim=True)+1e-6)

        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)

        return self.backbone(x)

## EfficientNet Training 

In [33]:
run=wandb.init(
    project=WANDB_PROJECT,
    name="efficientnet_augmented",
    reinit=True,
    config={
        "model":"EfficientNetB0",
        "augmentation":"mixup + specaugment",
        "optimizer":"AdamW",
        "epochs":EPOCHS
    }
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_092248-mtmh0b6a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run efficientnet_augmented
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/mtmh0b6a


In [34]:
effnet_model = EfficientNetAudio().to(device)

optimizer = torch.optim.AdamW(effnet_model.parameters(),lr=3e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)

criterion = nn.CrossEntropyLoss()

for epoch in range(30):

    effnet_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x,y = x.to(device),y.to(device)

        optimizer.zero_grad()

        preds = effnet_model(x)

        loss = criterion(preds,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    run.log({
        "epoch": epoch,
        "train_loss": avg_loss,
        "lr": optimizer.param_groups[0]["lr"]
    })

    scheduler.step()

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

100%|██████████| 15/15 [01:34<00:00,  6.33s/it]


Epoch 0 Loss 2.045881994565328


100%|██████████| 15/15 [01:34<00:00,  6.33s/it]


Epoch 1 Loss 1.4064552466074625


...

Epoch 27 Loss 0.6835234542687734


100%|██████████| 15/15 [01:35<00:00,  6.39s/it]


Epoch 28 Loss 0.6617848197619121


100%|██████████| 15/15 [01:35<00:00,  6.39s/it]

Epoch 29 Loss 0.4822800646225611


## EfficientNet Evalution

In [35]:
f1, acc = evaluate_model(effnet_model)

run.log({
    "F1_score": f1,
    "accuracy": acc
})

run.finish()

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:   F1_score ▁
wandb:   accuracy ▁
wandb:      epoch ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:         lr ██████▇▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁▁▁▁
wandb: train_loss █▅▄▄▃▃▃▃▄▃▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▁▂▂▂▁
wandb: 
wandb: Run summary:
wandb:   F1_score 0.8476
wandb:   accuracy 0.85
wandb:      epoch 29
wandb:         lr 0.0
wandb: train_loss 0.48228
wandb: 
wandb: 🚀 View run efficientnet_augmented at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/mtmh0b6a
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_092248-mtmh0b6a/logs


# Final EfficientNet Model (Advanced Augmentation)
## Enhanced Audio Classifier

In [36]:
class AudioClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.freq_mask = torchaudio.transforms.FrequencyMasking(48)
        self.time_mask = torchaudio.transforms.TimeMasking(96)

        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    def forward(self,x):

        # x = x.squeeze(1)
        x = self.mel(x)
        # x = x.unsqueeze(1)

        x = torch.log(x + 1e-6)

        x = (x - x.mean(dim=(2,3), keepdim=True)) / (x.std(dim=(2,3), keepdim=True) + 1e-6)

        if self.training:
            if random.random() < 0.5:
                x = self.freq_mask(x)
            if random.random() < 0.5:
                x = self.time_mask(x)

        return self.backbone(x)

## Mixup Training Strategy

In [37]:
run = wandb.init(
    project=WANDB_PROJECT,
    name="efficientnet_final_augmented",
    reinit=True,
    config={
        "model": "EfficientNetB0",
        "augmentation": "Mixup + SpecAugment + T-Stretch",
        "epochs": EPOCHS,
        "lr": 3e-4
    }
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_101103-sx6txlqe
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run efficientnet_final_augmented
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/sx6txlqe


In [38]:
effnet_aug_model = AudioClassifier().to(device)

optimizer = torch.optim.AdamW(effnet_aug_model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.CrossEntropyLoss()

## Final Model Training

In [39]:
for epoch in range(EPOCHS):

    effnet_aug_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        lam = np.random.beta(0.4, 0.4)

        index = torch.randperm(x.size(0)).to(device)
        
        mixed_x = lam * x + (1 - lam) * x[index]
        
        y_a, y_b = y, y[index]
        
        preds = effnet_aug_model(mixed_x)
        
        loss = lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(effnet_aug_model.parameters(), 1.0)
        
        optimizer.step()

        total_loss += loss.item()
    scheduler.step()
    run.log({"epoch": epoch, "loss": total_loss/len(train_loader)})

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

100%|██████████| 15/15 [01:36<00:00,  6.44s/it]


Epoch 0 Loss 2.372613875071208


100%|██████████| 15/15 [01:35<00:00,  6.39s/it]


Epoch 1 Loss 1.7280131498972575


100%|██████████| 15/15 [01:37<00:00,  6.48s/it]


...

Epoch 37 Loss 1.1444681882858276


100%|██████████| 15/15 [01:36<00:00,  6.43s/it]


Epoch 38 Loss 0.9360915044943492


100%|██████████| 15/15 [01:36<00:00,  6.45s/it]

Epoch 39 Loss 0.996923158566157


## Validation Evaluation

In [40]:
def evaluate():

    effnet_aug_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x,y in val_loader:

            x = x.to(device)
            y = y.to(device)

            preds = effnet_aug_model(x)

            preds = preds.argmax(1)

            correct += (preds==y).sum().item()
            total += len(y)

    return correct/total

In [41]:
val_acc = evaluate()
print("Validation Accuracy:",val_acc)
run.log({"final_val_accuracy": val_acc})
run.finish()

wandb: updating run metadata


Validation Accuracy: 0.88


wandb: 
wandb: Run history:
wandb:              epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb: final_val_accuracy ▁
wandb:               loss █▅▆▄▄▄▃▃▃▄▃▃▄▃▃▂▂▂▂▁▂▂▁▂▂▂▃▂▂▂▁▂▁▂▁▁▂▂▁▁
wandb: 
wandb: Run summary:
wandb:              epoch 39
wandb: final_val_accuracy 0.88
wandb:               loss 0.99692
wandb: 
wandb: 🚀 View run efficientnet_final_augmented at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/sx6txlqe
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_101103-sx6txlqe/logs


# Test Dataset Preparation

In [42]:
class MashupDataset(Dataset):

    def __init__(self,test_df):

        self.df = test_df

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):

        row = self.df.iloc[idx]

        path = os.path.join(DATA_DIR,row.filename)

        y,_ = librosa.load(path,sr=22050)

        y = torch.tensor(y).float().unsqueeze(0)

        return y,row.id

## Test Data Loader

In [43]:
test_df = pd.read_csv(TEST_CSV)

test_ds = MashupDataset(test_df)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda b: collate_fn([(x,0) for x,_ in b])
)

# Ensemble Prediction
## Audio Prediction Function

In [44]:
def predict_audio(audio):

    crop_size = 22050 * 6

    crops = []

    for start in range(0, len(audio) - crop_size, crop_size // 4):
        crops.append(audio[start:start+crop_size])

    if len(crops) == 0:
        crops.append(random_crop(audio,crop_size))

    preds = []

    for crop in crops:

        x = torch.tensor(crop).float().unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():

            p1 = torch.softmax(cnn_model(x),1)
            p2 = torch.softmax(crnn_model(x),1)
            p3 = torch.softmax(effnet_model(x),1)
            p4 = torch.softmax(effnet_aug_model(x),1)

            p = (
                0.15*p1 +
                0.25*p2 +
                0.35*p3 +
                0.25*p4
            )

        preds.append(p.cpu().numpy())

    preds = np.mean(preds,axis=0)

    return preds.argmax()

## Ensemble Run Initialization

In [45]:
ensemble_run = wandb.init(
    project=WANDB_PROJECT, 
    name="final_ensemble_submission",
    reinit=True,
)

wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260315_111605-b3hgvedo
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run final_ensemble_submission
wandb: ⭐️ View project at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: 🚀 View run at https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/b3hgvedo


## Generating Predictions

In [46]:
cnn_model.eval()
crnn_model.eval()
effnet_model.eval()
effnet_aug_model.eval()

predictions = []

for i,row in tqdm(test_df.iterrows(), total=len(test_df)):

    path = os.path.join(DATA_DIR, row.filename)

    y,_ = librosa.load(path, sr=22050)

    pred = predict_audio(y)

    predictions.append(pred)
    
    if i % 100 == 0:
        ensemble_run.log({"progress": i / len(test_df)})

100%|██████████| 3020/3020 [17:36<00:00,  2.86it/s]


# Submission

In [47]:
submission = pd.read_csv(SUBMISSION_CSV)

submission["genre"] = [IDX_TO_GENRE[p] for p in predictions]

submission.head()

,id,genre
0,1,pop
1,2,classical
2,3,disco
3,4,metal
4,5,country


In [48]:
submission.to_csv("submission.csv",index=False)
ensemble_run.finish()

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb: progress ▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
wandb: 
wandb: Run summary:
wandb: progress 0.99338
wandb: 
wandb: 🚀 View run final_ensemble_submission at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026/runs/b3hgvedo
wandb: ⭐️ View project at: https://wandb.ai/23f3003805-dl-genai-project/23f3003805-t12026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260315_111605-b3hgvedo/logs
